In [1]:
# === CELL 1: STABLE INSTALLATION ===
import os
import subprocess

print("⏳ Installing Stable Unsloth & Dependencies...")

# 1. Install Unsloth and Unsloth-Zoo
subprocess.run('pip install --upgrade --no-cache-dir unsloth unsloth-zoo', shell=True)

# 2. Install Hugging Face dependencies
subprocess.run('pip install --no-deps xformers trl peft accelerate bitsandbytes', shell=True)

print("✅ Installation Complete. You can proceed directly to Cell 2!")

⏳ Installing Stable Unsloth & Dependencies...
✅ Installation Complete. You can proceed directly to Cell 2!


In [2]:
# === CELL 2: TRAIN & SAVE (UNQUANTIZED LORA - A100 OPTIMIZED) ===
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# ================= CONFIGURATION =================
dataset_path = "train.jsonl"
TRAINING_STEPS = -1
SAMPLE_SIZE    = None
NUM_EPOCHS     = 3
# =================================================

# ── 1. Load Model (UNQUANTIZED) ──
print("Loading Llama-3 8B (Full 16-bit Precision)...")
max_seq_length = 4096
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b", # Changed: Removed the -bnb-4bit version
    max_seq_length = max_seq_length,
    dtype = torch.bfloat16,            # Changed: Explicitly force 16-bit brain float
    load_in_4bit = False,              # Changed: Disabled 4-bit quantization
)

# ── 2. LoRA Adapters ──
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# ── 3. Prompt Template ──
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    texts = []
    for inst, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        text = alpaca_prompt.format(inst, inp, out) + tokenizer.eos_token
        texts.append(text)
    return {"text": texts}

# ── 4. Load & Prepare Data ──
dataset = load_dataset("json", data_files=dataset_path, split="train")

if SAMPLE_SIZE is not None:
    dataset = dataset.select(range(min(SAMPLE_SIZE, len(dataset))))

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Training on {len(dataset)} examples.")

# ── 5. Training Arguments (UNQUANTIZED) ──
training_args = TrainingArguments(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_ratio       = 0.1,
    max_steps          = TRAINING_STEPS,
    num_train_epochs   = NUM_EPOCHS,
    learning_rate      = 1e-4,
    fp16 = False,
    bf16 = True,                         # A100 handles BF16 natively
    logging_steps      = 1,
    optim              = "paged_adamw_32bit", # Changed: Using 32-bit optimizer to remove all 8-bit math
    weight_decay       = 0.01,
    lr_scheduler_type  = "cosine",
    seed               = 3407,
    output_dir         = "outputs",
    dataloader_num_workers = 0,
)

# ── 6. Train ──
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = training_args,
)

torch.cuda.empty_cache()
trainer.train()

# ── 7. Save GGUF (UNQUANTIZED) ──
print("Converting to 16-bit GGUF (f16)...")
# Changed: Saving as a full 16-bit GGUF instead of heavily compressed q4_k_m
model.save_pretrained_gguf("thesis_unquantized_model", tokenizer, quantization_method = "f16")
print("DONE! Proceed to copy to Drive.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading Llama-3 8B (Full 16-bit Precision)...
==((====))==  Unsloth 2026.3.17: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

...


Unsloth: Copying 4 files from cache to `thesis_unquantized_model`:   0%|          | 0/4 [00:00<?, ?it/s]
Unsloth: Copying 4 files from cache to `thesis_unquantized_model`:  25%|██▌       | 1/4 [00:12<00:38, 12.82s/it]
Unsloth: Copying 4 files from cache to `thesis_unquantized_model`:  50%|█████     | 2/4 [00:28<00:29, 14.58s/it]
Unsloth: Copying 4 files from cache to `thesis_unquantized_model`:  75%|███████▌  | 3/4 [00:50<00:17, 17.77s/it]
Unsloth: Copying 4 files from cache to `thesis_unquantized_model`: 100%|██████████| 4/4 [00:57<00:00, 14.44s/it]


Successfully copied all 4 files from cache to `thesis_unquantized_model`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:00<00:00, 7989.15it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [00:51<00:00, 12.76s/it]


Unsloth: Merge process complete. Saved to `/content/thesis_unquantized_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['f16'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['thesis_unquantized_model_gguf/llama-3-8b.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into f16. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['thesis_unquantized_model_gguf/llama-3-8b.F16.gguf']
Unsloth: No Ollama template mapping found for model 'unsloth/llama-3-8b'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model thesis_unquantized_model_gguf/llama-3-8b.F16.gguf -p "why is the sky blue?"
DONE! Proceed to copy to Drive.


In [5]:
# === CELL 3: SAVE TO GOOGLE DRIVE ===
from google.colab import drive
import shutil
import os

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Define Paths
# Changed: The output file name changes because we used f16 quantization
source_path = "thesis_unquantized_model_gguf/llama-3-8b.F16.gguf"
destination_folder = "/content/drive/My Drive/Thesis_Backup"

# 3. Create Backup Folder
if not os.path.exists(destination_folder):
    os.makedirs(destination_folder)

# 4. Copy the GGUF file
destination_path = os.path.join(destination_folder, "llama3_unquantized_lora.gguf")

print(f"🚚 Copying massive 16GB model to Google Drive: {destination_path}...")
shutil.copy(source_path, destination_path)
print("✅ SAVED! You can now close this tab safely.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚚 Copying massive 16GB model to Google Drive: /content/drive/My Drive/Thesis_Backup/llama3_unquantized_lora.gguf...
✅ SAVED! You can now close this tab safely.
